# SO-101 ピック＆プレースの模倣学習 第2版：2本指グリッパ × 摩擦把持（ROS 2 × Gazebo × LeRobot）

**タスク**：テーブル上のランダムな位置に置かれた赤いブロック（剛体）を、SO-101 アームで掴んで青いトレイに入れる。

**第1版からの変更点**
- ハンド：SO-101 の片側だけが動くジョーを、**両側の指が開いて閉じる平行2本指グリッパ**に置き換え
- 把持：ブロックは指と衝突する**剛体**。固定（DetachableJoint）などの補助は使わず、**指で挟んだ摩擦力だけ**で持ち上げて運ぶ

**流れ**：環境構築 → シミュレーション構築 → 学習データ作成（IK エキスパート）→ LeRobot ACT 学習 → 学習済みポリシーで自律実行 → 提出用動画

### 2 つの Python を使い分けます（ここが要点）
Colab では `/usr/bin/python3` が **Colab 本体の 3.13** です。これは変更せず、ROS 2 側は `/usr/bin/python3.12` を **明示的に指定して** 使います。

| どこで動くか | Python | 用途 |
|---|---|---|
| Colab のノートブック本体 | 3.13（Colab 標準のまま） | LeRobot・PyTorch・学習・推論サーバ・動画生成 |
| Ubuntu 24.04 の Python 3.12 `/usr/bin/python3.12` | 3.12 | ROS 2 Jazzy（rclpy）の ROS ノード・`ros2` コマンド |

両者は別プロセスで動き、TCP でつながります。**condacolab の導入や、python3 のリンク差し替え、ライブラリの差し替えは行わないでください**（環境が壊れます）。

### 実行のしかた
1. 「ランタイム → ランタイムを接続解除して削除」で **まっさらなランタイム** にする
2. 「ランタイムのタイプを変更 → T4 GPU」
3. **上から順に 1 セルずつ** 実行（各セルは前のセルが終わっていないと止まるようにしてあります）

> 所要時間の目安：環境構築 15〜20 分、データ収集 15〜20 分、学習 40〜60 分、評価と動画 10 分

## 0. Drive のマウントと環境の確認

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
SAVE_DIR = '/content/drive/MyDrive/PAI_second'
os.makedirs(SAVE_DIR, exist_ok=True)
print('ノートブックの Python :', sys.version.split()[0], '(', sys.executable, ')')
!lsb_release -ds
!ls -l /usr/bin/python3 /usr/bin/python3.1* 2>&1
!/usr/bin/python3.12 --version 2>&1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 環境構築
### 1-1. LeRobot（ノートブック側 = Python 3.13）
「セッションを再起動」を求められた場合は再起動し、**0. をもう一度実行してから** 1-2 に進んでください。

In [ ]:
!pip install -q "lerobot[training]==0.6.1" "imageio[ffmpeg]"
import lerobot, torch
print('lerobot', lerobot.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

### 1-2. ROS 2 用の Python 3.12 の確認
`/usr/bin/python3`（= Colab 本体の 3.13）は **変更しません**。`/usr/bin/python3.12` があることだけ確認し、無ければ apt で追加します。
apt の更新時に python3 を呼ぶ `command-not-found` のフック（ROS とは無関係）は、失敗の原因になるので無効にします。

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
rm -f /etc/apt/apt.conf.d/50command-not-found
echo "/usr/bin/python3 -> $(readlink -f /usr/bin/python3)  (Colab 本体用。変更しません)"
if [ ! -x /usr/bin/python3.12 ]; then
  echo "python3.12 が無いので apt で追加します"
  apt-get update -qq
  apt-get install -y -qq python3.12
fi
/usr/bin/python3.12 -c "import sys; assert sys.version_info[:2] == (3, 12); print('ROS 用 Python OK:', sys.version.split()[0])"

### 1-3. ROS 2 Jazzy + Gazebo Harmonic + ros_gz（apt）
Ubuntu 24.04 の公式組み合わせ（ROS 2 Jazzy ⇔ Gazebo Harmonic）を ROS の apt リポジトリから入れます。失敗した場合は apt のログの末尾を表示します。

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
CODENAME=$(lsb_release -cs)
if [ "$CODENAME" = "noble" ]; then ROS_DISTRO=jazzy; GZ_PKGS="ros-jazzy-ros-gz"
elif [ "$CODENAME" = "jammy" ]; then ROS_DISTRO=humble; GZ_PKGS="ros-humble-ros-gzharmonic gz-harmonic"
  curl -sSL https://packages.osrfoundation.org/gazebo.gpg -o /usr/share/keyrings/pkgs-osrf-archive-keyring.gpg
  echo "deb [arch=amd64 signed-by=/usr/share/keyrings/pkgs-osrf-archive-keyring.gpg] http://packages.osrfoundation.org/gazebo/ubuntu-stable $CODENAME main" > /etc/apt/sources.list.d/gazebo-stable.list
else echo "未対応の Ubuntu: $CODENAME"; exit 1; fi
echo "Ubuntu $CODENAME -> ROS 2 $ROS_DISTRO"
echo $ROS_DISTRO > /content/ros_distro.txt
curl -sSL https://raw.githubusercontent.com/ros/rosdistro/master/ros.key -o /usr/share/keyrings/ros-archive-keyring.gpg
echo "deb [arch=amd64 signed-by=/usr/share/keyrings/ros-archive-keyring.gpg] http://packages.ros.org/ros2/ubuntu $CODENAME main" > /etc/apt/sources.list.d/ros2.list
apt-get update -qq > /tmp/apt_update.log 2>&1 || { tail -20 /tmp/apt_update.log; exit 1; }
echo "apt install 中（10〜15 分）..."
apt-get install -y -qq ros-$ROS_DISTRO-ros-base $GZ_PKGS python3-numpy xvfb mesa-utils libgl1-mesa-dri libegl1 fonts-noto-cjk ffmpeg zip \
  > /tmp/apt.log 2>&1 || { echo "=== apt 失敗。ログ末尾 ==="; tail -40 /tmp/apt.log; exit 1; }
echo "apt install 完了"

### 1-4. ROS 2 / Gazebo のコマンド実行ヘルパと動作確認

In [ ]:
import subprocess, os, sys, time, signal, glob
ROS_DISTRO = open('/content/ros_distro.txt').read().strip()
WS = '/content/pai_second'
SAVE_DIR = '/content/drive/MyDrive/PAI_second'
URDF = '/content/so101/so101_new_calib.urdf'
os.makedirs(f'{WS}/sim', exist_ok=True); os.makedirs(f'{WS}/learning', exist_ok=True)
PY_ROS = '/usr/bin/python3.12'   # ROS 2 ノード用（3.12）。/usr/bin/python3 は 3.13 なので使わない
ROS2 = f'{PY_ROS} /opt/ros/{ROS_DISTRO}/bin/ros2'   # ros2 コマンドも 3.12 で起動する
PHYSICS_FLAG = ''                # 1-4 の確認結果で必要なら設定

def ros_cmd(cmd):
    # ROS 側の子プロセスでは、ノートブック（3.13）の環境変数が混ざらないようにする
    return ("unset PYTHONPATH PYTHONHOME LD_PRELOAD; export PATH=/usr/bin:/bin:/usr/sbin:/sbin:$PATH; "
            f"export AMENT_PYTHON_EXECUTABLE={PY_ROS}; source /opt/ros/{ROS_DISTRO}/setup.bash; "
            f"export GZ_SIM_RESOURCE_PATH={WS}/sim/gen; {cmd}")

def run_ros(cmd, show=True):
    p = subprocess.run(['bash', '-c', ros_cmd(cmd)], capture_output=True, text=True)
    if show:
        print(p.stdout[-3000:], p.stderr[-3000:])
    return p

def bg(cmd, log, env_extra=None):
    env = dict(os.environ, **(env_extra or {}))
    return subprocess.Popen(['bash', '-c', ros_cmd(cmd)], stdout=open(log, 'w'), stderr=subprocess.STDOUT,
                            env=env, preexec_fn=os.setsid)

def kill(p):
    if p is not None and p.poll() is None:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM); time.sleep(1)

def need(*paths, hint=''):
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        raise RuntimeError(f'前の手順が終わっていません（{hint} を実行してください）: {missing}')

# --- 動作確認 ---
p = run_ros(f'{PY_ROS} -c "import sys, rclpy, numpy; print(\'rclpy OK on Python\', sys.version.split()[0])"; '
            f'which gz; gz sim --versions; {ROS2} pkg list | grep -E "^ros_gz_(bridge|sim)$"')
assert 'rclpy OK on Python 3.12' in p.stdout, 'rclpy がシステム Python 3.12 で読み込めません（1-2, 1-3 の出力を確認）'
assert 'ros_gz_bridge' in p.stdout, 'ros_gz_bridge が見つかりません（1-3 の出力を確認）'

# Gazebo の物理エンジン（DART）が読み込めるか、箱が落ちるだけの最小ワールドを 2000 ステップ回して確認
open('/content/smoke.sdf', 'w').write('''<?xml version="1.0"?><sdf version="1.9"><world name="smoke">
<plugin filename="gz-sim-physics-system" name="gz::sim::systems::Physics"/>
<model name="box"><pose>0 0 1 0 0 0</pose><link name="l"><collision name="c"><geometry><box><size>0.1 0.1 0.1</size></box></geometry></collision></link></model>
</world></sdf>''')

def gz_smoke(flag):
    q = run_ros(f'timeout 120 gz sim -s -r -v 3 --iterations 2000 {flag} /content/smoke.sdf', show=False)
    log = q.stdout + q.stderr
    errs = [l for l in log.splitlines() if '[Err]' in l]
    return q.returncode == 0 and not errs, errs + [f'exit code {q.returncode}']

ok, errs = gz_smoke('')
if not ok:
    print('既定の物理エンジン(DART)で問題あり:', *errs[-10:], sep='\n  ')
    PHYSICS_FLAG = '--physics-engine gz-physics-bullet-featherstone-plugin'
    ok, errs = gz_smoke(PHYSICS_FLAG)
    print('bullet-featherstone で再試行:', 'OK' if ok else errs[-10:])
assert ok, 'Gazebo が起動できません（上のエラーを貼ってください）'
print('Gazebo OK | physics =', PHYSICS_FLAG or 'dartsim (default)')

## 2. シミュレーション構築
### 2-1. ソースコード（Google Drive の `PAI_second_code` フォルダ、無ければ GitHub から取得）
- `so101_kin.py`：URDF を直接読む順運動学・逆運動学（numpy のみ）
- `build_sim.py`：URDF の改造（元のジョーを外し、掌＋左右の直動指を追加）、URDF→SDF 変換と後処理（衝突・摩擦の設定、関節コントローラ、関節状態パブリッシャ）、ワールド生成
- `so101_task_node.py`：ROS 2 ノード（データ収集 / ポリシー実行）
- `bridge.yaml`：ROS 2 ⇔ Gazebo のトピック対応

In [ ]:
import shutil
NAME = 'PAI_second_code'
cands = (glob.glob(f'/content/drive/MyDrive/{NAME}') or glob.glob(f'/content/drive/MyDrive/*/{NAME}') or glob.glob(f'/content/drive/MyDrive/*/*/{NAME}'))
if not cands:
    # Drive に無ければ GitHub の公開リポジトリから取得し、フラットなフォルダに並べる
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pai-claude-maty0505/second_model', '/content/second_model'], check=True)
    os.makedirs(f'/content/{NAME}', exist_ok=True)
    for sub in ['sim', 'learning', '.']:
        for f in glob.glob(f'/content/second_model/{sub}/*'):
            if os.path.isfile(f): shutil.copy(f, f'/content/{NAME}/')
    cands = [f'/content/{NAME}']
CODE_DIR = cands[0]; print('code from', CODE_DIR)
for f in ['so101_kin.py', 'build_sim.py', 'so101_task_node.py', 'bridge.yaml', 'snapshot.py']:
    shutil.copy(f'{CODE_DIR}/{f}', f'{WS}/sim/{f}')
for f in ['to_lerobot.py', 'policy_server.py', 'make_video.py']:
    shutil.copy(f'{CODE_DIR}/{f}', f'{WS}/learning/{f}')
shutil.copy(f'{CODE_DIR}/README.md', f'{WS}/README.md')
for root, _, files in os.walk(WS):
    print(root, sorted(files))

### 2-2. SO-101 の公式モデル（URDF・メッシュ）を取得してワールドを生成

In [ ]:
%%bash
set -e
mkdir -p /content/so101/assets && cd /content/so101
BASE=https://raw.githubusercontent.com/TheRobotStudio/SO-ARM100/main/Simulation/SO101
curl -sSfL -o so101_new_calib.urdf $BASE/so101_new_calib.urdf
for f in $(grep -o 'assets/[^"]*' so101_new_calib.urdf | sort -u); do curl -sSfL -o $f $BASE/$f; done
echo "meshes: $(ls assets | wc -l) files"

In [ ]:
need(f'{WS}/sim/build_sim.py', hint='2-1')
need(URDF, hint='2-2 の上のセル')
p = run_ros(f'cd {WS}/sim && {PY_ROS} build_sim.py --urdf {URDF} --out {WS}/sim/gen')
print(open(f'{WS}/sim/gen/so101_model/model.sdf').read().count('collide_bitmask'), 'collisions configured')
assert p.returncode == 0 and os.path.exists(f'{WS}/sim/gen/world.sdf'), 'ワールド生成に失敗しました（上の出力を確認）'

### 2-3. Gazebo をヘッドレス起動 + ros_gz_bridge
まず GPU（EGL）でのヘッドレス描画を試し、カメラ画像が来なければ Xvfb + ソフトウェア描画に自動で切り替えます。

In [ ]:
need(f'{WS}/sim/gen/world.sdf', hint='2-2')
# NVIDIA EGL を GLVND に登録（Colab では未登録のことがある）
if glob.glob('/usr/lib64-nvidia/libEGL_nvidia.so*') and not os.path.exists('/usr/share/glvnd/egl_vendor.d/10_nvidia.json'):
    os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
    open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json', 'w').write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

gz_proc = bridge_proc = xvfb_proc = None

def camera_alive(timeout=60):
    p = run_ros(f'timeout {timeout} gz topic -e -t /top_cam/image -n 1 | head -c 200', show=False)
    return 'width' in p.stdout

def start_sim(mode):
    global gz_proc, xvfb_proc
    kill(gz_proc); kill(xvfb_proc)
    subprocess.run('pkill -f "gz sim"; pkill -f Xvfb; true', shell=True); time.sleep(2)
    world = f'{WS}/sim/gen/world.sdf'
    if mode == 'egl':
        gz_proc = bg(f'gz sim -s -r --headless-rendering -v 3 {PHYSICS_FLAG} {world}', '/content/gz.log')
    else:
        xvfb_proc = subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1280x1024x24'], preexec_fn=os.setsid,
                                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        gz_proc = bg(f'gz sim -s -r -v 3 {PHYSICS_FLAG} {world}', '/content/gz.log',
                     {'DISPLAY': ':99', 'LIBGL_ALWAYS_SOFTWARE': '1'})
    time.sleep(10)
    return camera_alive()

def start_bridge():
    global bridge_proc
    kill(bridge_proc)
    bridge_proc = bg(f'{ROS2} run ros_gz_bridge parameter_bridge --ros-args -p config_file:={WS}/sim/bridge.yaml', '/content/bridge.log')
    time.sleep(5)

RENDER_MODE = None
for mode in ['egl', 'xvfb']:
    print('trying', mode, '...')
    if start_sim(mode):
        RENDER_MODE = mode; break
    print('\n'.join([l for l in open('/content/gz.log').read().splitlines() if '[Err]' in l or 'rror' in l][-15:]))
print('render mode =', RENDER_MODE)
assert RENDER_MODE, 'カメラ画像が取得できませんでした（上のエラー行と /content/gz.log を貼ってください）'
start_bridge()
run_ros(f'{ROS2} topic list')

### 2-4. 動作確認：関節指令を送って、両カメラの画像を確認（ここが一番の確認ポイント）

In [ ]:
assert RENDER_MODE and gz_proc.poll() is None, '2-3 を実行してください'
run_ros(f'cd {WS}/sim && {PY_ROS} snapshot.py --out /content/snap')
from PIL import Image; import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(Image.open('/content/snap/top_cam.ppm')); ax[0].set_title('policy camera (input to ACT)')
ax[1].imshow(Image.open('/content/snap/video_cam.ppm')); ax[1].set_title('presentation camera')
[a.axis('off') for a in ax]; plt.show()

### 2-5. 把持テスト：エキスパートで 3 回だけ実行（摩擦で持ち上がるかの確認）
両側の指でブロックを挟み、摩擦だけで持ち上げてトレイに入れられるかを確認します。`success=True` が出ればデータ収集に進めます。
下に「掴む直前・挟んだ直後・持ち上げ中・トレイ上・解放後」の記録用カメラ画像を並べます。

In [ ]:
assert RENDER_MODE and gz_proc.poll() is None and bridge_proc.poll() is None, '2-3 を実行してください'
subprocess.run('rm -rf /content/data/grasptest', shell=True)
p = run_ros(f'cd {WS}/sim && timeout 900 {PY_ROS} -u so101_task_node.py --mode collect --urdf {URDF} '
            f'--out /content/data/grasptest --episodes 3 --max_tries 3 --record_video 3 --seed 7', show=False)
print('\n'.join(l for l in (p.stdout + p.stderr).splitlines() if 'episode' in l or 'Error' in l or 'rror' in l)[-3000:])
import numpy as np, glob
from PIL import Image; import matplotlib.pyplot as plt
vids = sorted(glob.glob('/content/data/grasptest/demo_video_*.npz'))
if vids:
    v = np.load(vids[0])['video']
    idx = [22, 28, 36, 50, 62, 70]
    fig, ax = plt.subplots(1, len(idx), figsize=(20, 3.5))
    for a, i in zip(ax, idx):
        a.imshow(v[min(i, len(v) - 1)]); a.set_title(f'step {i}'); a.axis('off')
    plt.show()
else:
    print('成功エピソードがありません（ログを確認）')

## 3. 学習データ作成（ROS 2 ノード × IK エキスパート）
- ブロック位置を毎回ランダム化（x: 0.17〜0.25 m, y: −0.08〜0.05 m）
- エキスパートは逆運動学で「接近→下降→把持→持ち上げ→トレイ上へ→下降→解放→帰還」の関節軌道を生成（タイミングにも揺らぎを付与）
- 制御ループはカメラフレーム（シミュレーション時間 10 Hz）に同期：画像・関節角（状態）・関節目標（行動）を記録
- 成功したエピソードだけを保存

In [ ]:
assert RENDER_MODE and gz_proc.poll() is None and bridge_proc.poll() is None, '2-3 を実行してください'
N_EPISODES = 60
collect = bg(f'cd {WS}/sim && {PY_ROS} -u so101_task_node.py --mode collect --urdf {URDF} '
             f'--out /content/data/raw --episodes {N_EPISODES} --max_tries 120 --record_video 3', '/content/collect.log')
while collect.poll() is None:
    time.sleep(30)
    lines = [l for l in open('/content/collect.log').read().splitlines() if 'episode' in l]
    print(lines[-1] if lines else '...', flush=True)
print(open('/content/collect.log').read()[-1500:])
assert collect.returncode == 0, 'データ収集が途中で止まりました（上のログを貼ってください）'

In [ ]:
need('/content/data/raw/ep_0000.npz', hint='3. のデータ収集')
!python {WS}/learning/to_lerobot.py --raw /content/data/raw --root /content/data/so101_gz_parallel_grasp
!mkdir -p {SAVE_DIR} && cp -r /content/data {SAVE_DIR}/ && echo saved to Drive

In [ ]:
# データセットの中身を確認（LeRobotDataset として読み込み）
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import numpy as np, matplotlib.pyplot as plt
ds = LeRobotDataset('local/so101_gz_parallel_grasp', root='/content/data/so101_gz_parallel_grasp')
print(ds)
ep0 = [ds[i] for i in range(ds.meta.episodes[0]['length'])]
act = np.stack([f['action'].numpy() for f in ep0])
plt.figure(figsize=(10, 3)); plt.plot(act); plt.legend(ds.features['action']['names'], fontsize=8)
plt.title('episode 0 actions (joint targets rad / finger opening m)'); plt.show()

## 4. 学習：LeRobot ACT（ノートブック側 Python 3.13 / GPU）
- 入力：ポリシーカメラ画像（160×120）＋ 状態 6 次元（関節角 5 ＋ 指の開き幅）
- 出力：今後 20 ステップ分の関節目標＋指の開き幅（action chunk）、10 ステップごとに再推論
- ブロックの位置は与えていない → **ポリシーが画像から位置を読み取る必要がある**

In [ ]:
need('/content/data/so101_gz_parallel_grasp/meta/info.json', hint='3. の変換セル')
# 学習中は Colab の CPU（2 コア）をデータローダに回すため、シミュレータを一旦停止する
kill(bridge_proc); kill(gz_proc); kill(xvfb_proc)
TRAIN_STEPS = 10000   # T4 で約 70 分（第1版と同じ設定）
!lerobot-train \
  --dataset.repo_id=local/so101_gz_parallel_grasp --dataset.root=/content/data/so101_gz_parallel_grasp \
  --policy.type=act --policy.device=cuda --policy.push_to_hub=false \
  --policy.chunk_size=20 --policy.n_action_steps=10 \
  --output_dir=/content/outputs/act_second --job_name=act_second \
  --steps={TRAIN_STEPS} --batch_size=32 --num_workers=2 --log_freq=200 --save_freq=5000 \
  --wandb.enable=false 2>&1 | tee /content/train.log | grep --line-buffered -E "INFO.*step:|Checkpoint|Error|error"
!cp -r /content/outputs/act_second/checkpoints/last {SAVE_DIR}/act_second_last && cp /content/train.log {SAVE_DIR}/ && echo saved to Drive

## 5. 評価：学習済みポリシーで Gazebo のロボットを動かす
`policy_server.py`（ノートブック側 3.13・GPU）と `so101_task_node.py --mode eval`（ROS 2 側 3.12）を TCP でつなぎます。学習時に使っていない乱数シードでブロックを配置します。

In [ ]:
CKPT = '/content/outputs/act_second/checkpoints/last/pretrained_model'
need(f'{CKPT}/model.safetensors', hint='4. の学習')
# シミュレータを再起動（学習前に停止したため）
assert start_sim(RENDER_MODE), 'シミュレータの再起動に失敗しました'
start_bridge()

server = subprocess.Popen([sys.executable, f'{WS}/learning/policy_server.py', '--ckpt', CKPT, '--port', '5555'],
                          stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(90):
    time.sleep(2)
    if 'ready' in open('/content/server.log').read() or server.poll() is not None: break
print(open('/content/server.log').read()[-800:])
assert server.poll() is None, '推論サーバが起動できませんでした（上のログを貼ってください）'

N_EVAL = 10
ev = bg(f'cd {WS}/sim && {PY_ROS} -u so101_task_node.py --mode eval --urdf {URDF} '
        f'--out /content/data/eval --episodes {N_EVAL} --seed 12345 --port 5555 --max_steps 130', '/content/eval.log')
while ev.poll() is None:
    time.sleep(20)
    lines = [l for l in open('/content/eval.log').read().splitlines() if '[eval]' in l]
    print(lines[-1] if lines else '...', flush=True)
print('\n'.join(l for l in open('/content/eval.log').read().splitlines() if '[eval]' in l))
assert ev.returncode == 0, '評価が途中で止まりました（/content/eval.log を貼ってください）'

## 6. 提出用動画の生成（等倍・1〜3 分）

In [ ]:
need('/content/data/eval/eval_000.npz', hint='5. の評価')
# 全体版（成功例＋失敗例 1 つ）と、成功例だけの版の 2 本を作る
!python {WS}/learning/make_video.py --raw /content/data/raw --eval /content/data/eval --train_log /content/train.log --out /content/PAI_second.mp4
!python {WS}/learning/make_video.py --raw /content/data/raw --eval /content/data/eval --train_log /content/train.log --out /content/PAI_second_success.mp4 --only_success --n_eval 8
!ffprobe -v error -show_entries format=duration -of csv=p=0 /content/PAI_second.mp4
!ffprobe -v error -show_entries format=duration -of csv=p=0 /content/PAI_second_success.mp4
!cp /content/PAI_second.mp4 /content/PAI_second_success.mp4 /content/train.log /content/eval.log /content/collect.log {SAVE_DIR}/ && echo saved to Drive
from IPython.display import Video
Video('/content/PAI_second_success.mp4', embed=True, width=800)

## 7. GitHub 提出用にコード一式をまとめる
`MyDrive/PAI_second/pai_second_code.zip` に保存されます。このノートブック自体も「ファイル → ダウンロード → .ipynb」で取得し、zip の中身と一緒に GitHub の public リポジトリに置いてください。

In [ ]:
!cd /content && zip -qr {SAVE_DIR}/pai_second_code.zip pai_second -x "pai_second/sim/gen/*" -x "*__pycache__*" && ls -la {SAVE_DIR}

In [ ]:
# 後片付け
for name in ['server', 'bridge_proc', 'gz_proc', 'xvfb_proc']:
    p = globals().get(name)
    if p is not None and p.poll() is None:
        p.terminate() if name == 'server' else kill(p)
print('stopped')